# 数据集构造 — 可视化检查

检查流水线各步骤的输出是否正常。

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from visualize import (
    check_crop_samples,
    check_histogram_match,
    check_prediction,
)

## 1. 查看裁剪的振幅-相位样本

运行 `crop_dataset.py` 后，检查从参考复振幅中裁剪的 patch 是否合理。

In [ ]:
check_crop_samples(n=4, split="train")

## 2. 直方图匹配效果

对比自然图像灰度图 → 直方图匹配后的振幅图，以及它们的像素分布。

In [ ]:
check_histogram_match(n=4)

## 3. 查看 U-Net 预测结果

加载训练好的 checkpoint，对直方图匹配后的振幅推理相位。

需要先运行 `train_unet.py` 生成 `checkpoints/best.pt`。
如尚未训练，可将 `ckpt_name` 改为 `"last.pt"` 或跳过此步。

In [ ]:
# 如有 checkpoint 则展示预测结果
if os.path.exists("checkpoints/best.pt"):
    check_prediction(n=4, ckpt_name="best.pt")
elif os.path.exists("checkpoints/last.pt"):
    check_prediction(n=4, ckpt_name="last.pt")
else:
    print("未找到 checkpoint，请先运行 train_unet.py")

## 4. 查看最终数据集

运行 `generate_dataset.py` 后，检查生成的 `synth_dataset_v1`。

In [ ]:
import config

for split in ["train", "val", "test"]:
    amp_path = os.path.join(config.OUTPUT_DATASET_DIR, split, "label_amp.npy")
    phs_path = os.path.join(config.OUTPUT_DATASET_DIR, split, "label_phs.npy")

    if os.path.exists(amp_path):
        amp = np.load(amp_path)
        phs = np.load(phs_path)
        print(f"[{split}] amp: {amp.shape}, range=[{amp.min():.4f}, {amp.max():.4f}]")
        print(f"[{split}] phs: {phs.shape}, range=[{phs.min():.4f}, {phs.max():.4f}]")

        # 展示 4 组
        idxs = np.random.choice(len(amp), 4, replace=False)
        fig, axes = plt.subplots(4, 2, figsize=(8, 10))
        for row, idx in enumerate(idxs):
            axes[row, 0].imshow(amp[idx], cmap="gray")
            axes[row, 0].set_title(f"Amp ({idx})")
            axes[row, 0].axis("off")
            axes[row, 1].imshow(phs[idx], cmap="gray")
            axes[row, 1].set_title(f"Phase ({idx})")
            axes[row, 1].axis("off")
        plt.suptitle(f"Split: {split}")
        plt.tight_layout()
        plt.show()
    else:
        print(f"[{split}] 未找到，请先运行 generate_dataset.py")